# Wave Packets, Group Velocity and Uncertainty Relations

## Step 0 – Imports and constants

We use SI units internally. Lengths will be plotted in **nm**, wave numbers in **nm**$^{-1}$, and time in **fs** whenever convenient.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt

h = 6.62607015e-34            # J s
hbar = 1.054571817e-34        # J s
m_e = 9.1093837015e-31        # kg
eV = 1.602176634e-19          # J

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

print('Imports succesful.')
print(f'hbar = {hbar:.6e} J s')
print(f'm_e = {m_e:.6e} kg')

/home/jp/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Imports succesful.
hbar = 1.054572e-34 J s
m_e = 9.109384e-31 kg


## Step 1 – Grid and numerical parameters

We work on a one-dimensional grid centered at $x=0$. The corresponding $k$ grid is obtained from the FFT convention.


In [2]:
N = 512
x_max = 40e-9         # 40 nm on each side
dx = 2 * x_max / N
x = (np.arange(N) - N // 2) * dx
L = N * dx
k = 2 * np.pi * np.fft.fftfreq(N, d=dx)
dk = 2 * np.pi / L

lambda0 = 1.6e-9      # central de Broglie wavelength (m)
k0 = 2 * np.pi / lambda0
sigma_k0 = 0.30e9     # 1/m
Delta_k_two = 0.25e9  # 1/m

print(f'N = {N}')
print(f'dx = {dx:.3e} m')
print(f'L  = {L:.3e} m')
print(f'dk = {dk:.3e} 1/m')
print(f'Central wavelength = {lambda0*1e9:.3f} nm')
print(f'k0 = {k0:.3} 1/m')


N = 512
dx = 1.563e-10 m
L  = 8.000e-08 m
dk = 7.854e+07 1/m
Central wavelength = 1.600 nm
k0 = 3.93e+09 1/m


In [4]:
(2.273e05*2*m_e)/hbar

3926831500.6581483

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt

h = 6.62607015e-34            # J s
hbar = 1.054571817e-34        # J s
m_e = 9.1093837015e-31        # kg
eV = 1.602176634e-19          # J

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

print('Imports succesful.')
print(f'hbar = {hbar:.6e} J s')
print(f'm_e = {m_e:.6e} kg')


Imports succesful.
hbar = 1.054572e-34 J s
m_e = 9.109384e-31 kg


In [6]:
N = 512
x_max = 40e-9         # 40 nm on each side
dx = 2 * x_max / N
x = (np.arange(N) - N // 2) * dx
L = N * dx
k = 2 * np.pi * np.fft.fftfreq(N, d=dx)
dk = 2 * np.pi / L

lambda0 = 1.6e-9      # central de Broglie wavelength (m)
k0 = 2 * np.pi / lambda0
sigma_k0 = 0.30e9     # 1/m
Delta_k_two = 0.25e9  # 1/m

print(f'N = {N}')
print(f'dx = {dx:.3e} m')
print(f'L  = {L:.3e} m')
print(f'dk = {dk:.3e} 1/m')
print(f'Central wavelength = {lambda0*1e9:.3f} nm')
print(f'k0 = {k0:.3} 1/m')


N = 512
dx = 1.563e-10 m
L  = 8.000e-08 m
dk = 7.854e+07 1/m
Central wavelength = 1.600 nm
k0 = 3.93e+09 1/m


In [7]:
def normalize_on_grid(psi_x, dx):
  norm = np.sqrt(np.sum(np.abs(psi_x)**2)*dx)
  return psi_x/norm

def x_to_k(psi_x, dx):
  psi_shifted = np.fft.ifftshift(psi_x)
  phi_k = (dx / np.sqrt(2*np.pi)) * np.fft.fft(psi_shifted)
  return phi_k

def k_to_x(phi_k, dk):
  Nloc = len(phi_k)
  psi_x = np.fft.fftshift(np.fft.ifft(phi_k)*Nloc*dk/(np.sqrt(2*np.pi)))
  return psi_x

def omega_free(k):
  return hbar*k**2/(2*m_e)

def plane_wave(x,k0,t=0.0,A=1.0):
  phase = k0*x-omega_free(k0)*t
  psi = A*np.exp(1j*phase)
  return psi

def two_wave_superposition(x,t,k0,Delta_k,A=0.5):
  k1=k0-Delta_k/2
  k2=k0+Delta_k/2
  w1=omega_free(k1)
  w2=omega_free(k2)
  psi1=A*np.exp(1j*(k1*x-w1*t))
  psi2=A*np.exp(1j*(k2*x-w2*t))
  psi_total=psi1+psi2
  return psi_total,k1,k2,w1,w2

def gaussian_phi_k(k,k0,sigma_k,dk):
  phi=np.exp(-0.5*((k-k0)/sigma_k)**2)
  norm=np.sqrt(np.sum(np.abs(phi)**2)*dk)
  phi=phi/norm
  return phi

def rectangular_phi_k(k,k0,width,dk):
  phi=np.where(np.abs(k-k0)<=width/2,1.0,0.0)
  phi=phi.astype(complex)  
  norm=np.sqrt(np.sum(np.abs(phi)**2)*dk)
  phi=phi/norm
  return phi

def evolve_free_from_phi(phi_k,t,k,dk,dx):
  phase_factor=np.exp(-1j*omega_free(k)*t)
  phi_k_t=phi_k*phase_factor
  phi_t=k_to_x(phi_k_t,dk)
  psi_t=normalize_on_grid(psi_t,dx)
  return psi_t

def k_moments_numeric(phi_k,k,dx):
  rho_k=np.abs(phi_k)**2
  k_mean=np.sum(k*rho_k)*dk
  # k2_mean=np.sum(k**2*rho_k)*dk
  # variance_k=np.maximum(k2_mean-(k_mean)**2,0.0)
  # Dk=np.sqrt(variance_k)
  variance_k = np.sum((k - k_mean)**2 * rho_k) * dk
  Dk = np.sqrt(np.maximum(variance_k, 0.0))
  return k_mean,Dk

def ket_from_psi(psi_x,dx):
  ket_vector=psi_x*np.sqrt(dx)
  ket_vector=ket_vector.reshape(-1,1)
  ket=qt.Qobj(ket_vector)
  return ket

def x_moments_numeric(psi_x,x,dx):
  rho=np.abs(psi_x)**2
  x_mean=np.sum(x*rho)*dx
  x2_mean=np.sum(x**2*rho)*dx
  variance_x=np.maximum(x2_mean-(x_mean)**2,0.0)
  Dx=np.sqrt(variance_x)
  return x_mean,Dx

def build_x_operator(x):
  x_matrix=np.diag(x)
  x_matrix=x_matrix.astype(complex)
  x_operator=qt.Qobj(x_matrix)
  return x_operator

def expectation_and_variance(ket,op):
  mean=np.real(qt.expect(op, ket))
  mean2=np.real(qt.expect(op**2,ket))
  variance=np.maximum(mean2-mean**2,0.0)
  std_dev=np.sqrt(variance)
  return mean,std_dev


In [8]:
sigma_scan = np.array([0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.65]) * 1e9


In [10]:
phi = gaussian_phi_k(k,k0,0.15,dk)

In [14]:
k_moments_numeric(phi,k,dk)

(np.float64(3926990816.98724), np.float64(1.4305114746093748e-06))

In [12]:
psi = normalize_on_grid(k_to_x(phi,dk),dx)

In [ ]:
ket = ket_from_psi(psi,dx)

Quantum object: dims=[[512], [1]], shape=(512, 1), type='ket', dtype=Dense
Qobj data =
[[ 0.04419417+0.j        ]
 [ 0.03613249+0.02544737j]
 [ 0.01488857+0.04161076j]
 [-0.01178715+0.04259329j]
 [-0.03416256+0.02803649j]
 [-0.04407443+0.00325113j]
 [-0.03790661-0.02272035j]
 [-0.01790931-0.04040274j]
 [ 0.00862186-0.043345j  ]
 [ 0.0320075 -0.03047367j]
 [ 0.04371584-0.00648463j]
 [ 0.03947531+0.0198702j ]
 [ 0.02083299+0.03897578j]
 [-0.00540984+0.04386181j]
 [-0.02967899+0.03274572j]
 [-0.04312035+0.009683j  ]
 [-0.04083009-0.01691238j]
 [-0.02364378-0.03733761j]
 [ 0.00216851-0.04414094j]
 [ 0.02718965-0.03484032j]
 [ 0.04229119-0.01282889j]
 [ 0.04196361+0.01386291j]
 [ 0.02632644+0.03549709j]
 [ 0.00108458+0.04418086j]
 [-0.02455297+0.03674611j]
 [-0.04123285+0.01590526j]
 [-0.04286973-0.01073831j]
 [-0.02886643-0.03346422j]
 [-0.00433179-0.04398137j]
 [ 0.02178323-0.03845278j]
 [ 0.03995106-0.01889544j]
 [ 0.04354353+0.00755552j]
 [ 0.03125   +0.03125j   ]
 [ 0.00755552+0.043543

In [19]:
from scipy.linalg import dft
from scipy.sparse import diags

x_op = build_x_operator(x)
def build_p_operator(N, dx):
    ones = np.ones(N - 1)
    diff_matrix=diags([ones,-ones],[1,-1],shape=(N,N)).toarray()/(2*dx)
    diff_matrix[0, -1] = -1 / (2 * dx)
    diff_matrix[-1, 0] = 1 / (2 * dx)
    p_matrix = -1j * hbar * diff_matrix
    return qt.Qobj(p_matrix)
p_op = build_p_operator(N, dx)
sigma_scan = np.array([0.15]) * 1e9
Dx_list = []
Dk_list = []
Dp_list = []
for sig in sigma_scan:
  phi = gaussian_phi_k(k,k0,sig,dk)
  psi = normalize_on_grid(k_to_x(phi,dk),dx)
  ket = ket_from_psi(psi,dx)
  _,Dx_here = expectation_and_variance(ket,x_op)
  _,Dp_here = expectation_and_variance(ket,p_op)
  _,Dk_here = k_moments_numeric(phi,k,dk)
  Dx_list.append(Dx_here)
  Dp_list.append(Dp_here)
  Dk_list.append(Dk_here)

In [20]:
Dx_list

[np.float64(4.714045207910378e-09)]

In [21]:
Dp_list

[np.float64(9.144087854978855e-27)]

In [26]:
Dk_list

[np.float64(106066017.17798035)]

In [27]:
for sig,dxv,dkv,dpv in zip(sigma_scan,Dx_list,Dk_list,Dp_list):
  print(f'sigma_k = {sig*1e-9:.3f} nm^-1'
  f'Delta x = {dxv*1e9:.3f} nm'
  f'Delta k = {dkv*1e-9:.3f} nm^-1'
  f'Delta x * Delta p / hbar = {dxv*dpv/hbar:.3f}')


sigma_k = 0.150 nm^-1Delta x = 4.714 nmDelta k = 0.106 nm^-1Delta x * Delta p / hbar = 0.409
